
# Project 03 — Credit Risk: PD / LGD / EAD & Expected Loss
## Notebook 05 — Portfolio Credit Risk & Stress Testing

This notebook extends the expected-loss analysis from Notebook 04 by summarizing the overall portfolio risk profile and examining the sensitivity of expected credit losses to stressed default probabilities.

**Objectives**

- Summarize portfolio exposure, average PD, weighted-average PD, and average LGD.
- Quantify total expected loss and the expected-loss-to-exposure ratio.
- Evaluate moderate and severe stress scenarios using transparent PD multipliers.
- Compare expected losses across scenarios and quantify the incremental impact of stress.
- Interpret the economic implications and limitations of the results.

**Methodology**

Expected loss is calculated using:

$$
EL_i = PD_i \times LGD_i \times EAD_i
$$

Stress scenarios increase loan-level PD while holding LGD and EAD constant. This isolates the effect of higher default probabilities on portfolio expected loss.

The analysis uses simplified, assumption-based credit risk inputs established in the preceding notebooks. The stress scenarios are illustrative sensitivity analyses, not macroeconomic forecasts or estimates of losses under a calibrated economic downturn.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("../data/processed")

train_df = pd.read_csv(data_dir / "credit_risk_development_el.csv")
valid_df = pd.read_csv(data_dir / "credit_risk_validation_el.csv")



## 1. Portfolio Risk Profile

The portfolio risk profile consolidates loan-level credit risk estimates into a set of portfolio-level metrics. These measures summarize the portfolio's total exposure, default risk, loss severity, and expected credit losses.

The analysis includes the following metrics:

- **Total Exposure (EAD):** The sum of exposure at default across all loans.
- **Average PD:** The arithmetic mean of the probability of default across loans.
- **Weighted-Average PD:** The average probability of default weighted by each loan's exposure, giving larger exposures greater influence.
- **Average LGD:** The arithmetic mean of loss given default across loans.
- **Total Expected Loss:** The sum of expected losses across all loans, calculated as the product of PD, LGD, and EAD.
- **EL / EAD:** Total expected loss divided by total exposure, expressing expected loss as a proportion of the portfolio's exposure.

For a portfolio of \(n\) loans, total expected loss is:

$$
EL_{portfolio} = \sum_{i=1}^{n} PD_i \times LGD_i \times EAD_i
$$

The weighted-average probability of default is:

$$
PD_{weighted} =
\frac{\sum_{i=1}^{n} PD_i \times EAD_i}
{\sum_{i=1}^{n} EAD_i}
$$

These metrics provide a consolidated view of portfolio credit risk and establish the baseline against which the subsequent stress scenarios will be compared.


In [2]:

portfolio_df = train_df.copy()

total_ead = portfolio_df["ead_proxy"].sum()

average_pd = portfolio_df["predicted_pd"].mean()

weighted_average_pd = (
    (portfolio_df["predicted_pd"] * portfolio_df["ead_proxy"]).sum()
    / total_ead
)

average_lgd = portfolio_df["lgd_assumption"].mean()

total_expected_loss = portfolio_df["expected_loss"].sum()

el_to_ead = total_expected_loss / total_ead

portfolio_metrics = pd.DataFrame({
    "Metric": [
        "Total Exposure (EAD)",
        "Average PD",
        "Weighted-Average PD",
        "Average LGD",
        "Total Expected Loss",
        "EL / EAD"
    ],
    "Value": [
        total_ead,
        average_pd,
        weighted_average_pd,
        average_lgd,
        total_expected_loss,
        el_to_ead
    ]
})

portfolio_metrics


,Metric,Value
0,Total Exposure (EAD),2.159512e+10
1,Average PD,1.877075e-01
2,Weighted-Average PD,2.021069e-01
3,Average LGD,4.634913e-01
4,Total Expected Loss,2.200969e+09
5,EL / EAD,1.019197e-01



### 1.1 Interpretation

The portfolio risk profile provides a consolidated view of the credit risk estimates generated in the preceding notebooks.

The total exposure represents the aggregate exposure at default across the portfolio, while total expected loss estimates the losses implied by the loan-level PD, LGD, and EAD assumptions.

The weighted-average PD is higher than the arithmetic average PD, indicating that loans with larger exposures have a greater influence on the portfolio's overall default risk than they would under an equally weighted calculation.

The EL / EAD ratio expresses expected losses relative to total exposure, providing a normalized measure that can be compared across portfolios of different sizes.

These results establish the baseline portfolio risk profile for the stress-testing analysis that follows.



## 2. Stress Testing

Stress testing examines how the portfolio's expected credit losses change when borrowers become more likely to default.

Three scenarios are considered:

- **Base Case:** Uses the original predicted probabilities of default.
- **Moderate Stress:** Increases each loan's PD by 25%.
- **Severe Stress:** Increases each loan's PD by 50%.

The stressed probability of default is calculated as:

$$
PD_i^{stress} = \min(PD_i \times m, 1)
$$

where \(m\) is the scenario multiplier. The probability is capped at 1 to ensure it remains within the valid range.

Expected loss under each scenario is calculated as:

$$
EL_{stress} =
\sum_{i=1}^{n} PD_i^{stress} \times LGD_i \times EAD_i
$$

LGD and EAD remain unchanged across scenarios. This isolates the effect of increasing default probabilities on expected losses.

The analysis compares total expected loss and the EL / EAD ratio across scenarios. The increase in expected loss relative to the base case illustrates the portfolio's sensitivity to deterioration in credit quality.

These scenarios are illustrative sensitivity analyses rather than forecasts of actual future losses.


In [3]:

scenarios = {
    "Base Case": 1.00,
    "Moderate Stress": 1.25,
    "Severe Stress": 1.50
}


In [4]:

stress_results = []

for scenario, multiplier in scenarios.items():
    stressed_pd = (portfolio_df["predicted_pd"] * multiplier).clip(upper=1.0)

    stressed_el = (
        stressed_pd
        * portfolio_df["lgd_assumption"]
        * portfolio_df["ead_proxy"]
    ).sum()

    stress_results.append({
        "Scenario": scenario,
        "PD Multiplier": multiplier,
        "Weighted-Average PD": (
            (stressed_pd * portfolio_df["ead_proxy"]).sum()
            / total_ead
        ),
        "Average LGD": average_lgd,
        "Total EAD": total_ead,
        "Expected Loss": stressed_el,
        "EL / EAD": stressed_el / total_ead
    })

stress_results_df = pd.DataFrame(stress_results)

base_el = stress_results_df.loc[
    stress_results_df["Scenario"] == "Base Case",
    "Expected Loss"
].iloc[0]

stress_results_df["Incremental EL"] = (
    stress_results_df["Expected Loss"] - base_el
)

stress_results_df


,Scenario,PD Multiplier,Weighted-Average PD,Average LGD,Total EAD,Expected Loss,EL / EAD,Incremental EL
0,Base Case,1.00,0.202107,0.463491,2.159512e+10,2.200969e+09,0.101920,0.000000e+00
1,Moderate Stress,1.25,0.252613,0.463491,2.159512e+10,2.750981e+09,0.127389,5.500122e+08
2,Severe Stress,1.50,0.303098,0.463491,2.159512e+10,3.300740e+09,0.152847,1.099771e+09



### 2.1 Interpretation

The stress-testing results demonstrate the sensitivity of portfolio expected losses to increases in the probability of default.

Under the moderate stress scenario, the weighted-average PD increases from 20.21% to 25.26%, raising total expected loss from approximately $2.20 billion to $2.75 billion. This represents an incremental expected loss of approximately $550 million.

Under the severe stress scenario, the weighted-average PD rises to 30.31%, and total expected loss increases to approximately $3.30 billion. The incremental expected loss relative to the base case is approximately $1.10 billion.

Because LGD and EAD remain unchanged, the increase in expected loss is driven entirely by the higher probabilities of default. The EL / EAD ratio also rises, indicating a greater expected loss relative to the portfolio's total exposure.

These results illustrate how deterioration in borrower credit quality can materially increase expected credit losses. However, the scenarios are simplified sensitivity analyses: they do not model correlations between defaults, changes in recovery rates, changes in exposure, or the effects of specific macroeconomic conditions.



## 3. Conclusion

This notebook consolidated loan-level credit risk estimates into a portfolio risk profile and evaluated the impact of moderate and severe increases in default probabilities.

The analysis demonstrates that portfolio expected losses can increase substantially under adverse credit conditions, even when loss severity and exposure remain unchanged.

Together, the preceding notebooks establish a credit risk workflow covering probability-of-default estimation, loss-given-default and exposure assumptions, expected-loss calculation, and portfolio-level stress testing.

The principal limitation of the stress analysis is its simplified design. The scenarios isolate changes in PD rather than modelling a comprehensive economic downturn. The results should therefore be interpreted as illustrative sensitivity estimates rather than forecasts of realized credit losses.
